
# Composable AGN: mix-and-match recipes

The composable AGN block subsystem (``agn_model="composable"``) lets users
pick one block per pipeline stage and combine across models. This example
compares four recipes built from the registered block set:

1. **All-GRAHSP** — Buchner+ 2024 end-to-end (SBPL BBB + Netzer lines +
   FeII forest + log-Gaussian torus + bi-attenuation).
2. **All-QSOgen** — Temple+ 2021 monolithic recipe expressed as five
   blocks (continuum + emission lines + Balmer + hot dust + SMC).
3. **GRAHSP BBB + SKIRTOR torus + SMC** — UV/optical from Buchner,
   mid-IR from Stalevski clumpy templates, attenuation by Pei SMC.
4. **Multicolor disc + Nenkova torus** — Shakura-Sunyaev disc with the
   CLUMPY (Nenkova+ 2008) radiative-transfer torus.

All four are evaluated through ``composable_agn_l_nu`` on the same
wavelength grid; the headline is that the user only changes selector
strings between them, not the call site.


In [ ]:
# TODO: refactor to SEDModel.build API (currently uses low-level internal API)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from tengri.analysis.plotting import setup_style
from tengri.components.agn.blocks import composable_agn_l_nu

setup_style()

wave_aa = jnp.logspace(np.log10(500.0), np.log10(1.0e6), 1000)
wave_um = np.asarray(wave_aa) / 1e4

# Common scaling — every recipe is evaluated at the same bolometric luminosity.
# Note: ``agn_log_lbol = 12`` means log10(L_bol / L_sun) = 12 → L_bol ≈ 4e45
# erg/s, a luminous-quasar value. All recipes auto-normalise from this; we do
# *not* pass ``agn_grahsp_l5100`` explicitly so GRAHSP uses the same bolometric
# anchor instead of bypassing it with a hard-coded scale.
COMMON = dict(agn_log_lbol=12.0, agn_frac=1.0)

# Each entry: (label, color, kwargs to override defaults).
RECIPES = [
    (
        "all-GRAHSP",
        "tab:blue",
        dict(
            agn_disc_block="grahsp_sbpl",
            agn_lines_block="grahsp",
            agn_feii_block="grahsp",
            agn_torus_block="grahsp",
            agn_attenuation_block="grahsp_biatten",
            agn_grahsp_a_feii=5.0,
            agn_grahsp_fcov=0.4,
            agn_grahsp_ebv=0.0,
            agn_grahsp_ebv_agn=0.0,
        ),
    ),
    (
        "all-QSOgen",
        "tab:orange",
        dict(
            agn_disc_block="qsogen",
            agn_lines_block="qsogen",
            agn_feii_block="qsogen_balmer",
            agn_torus_block="qsogen",
            agn_attenuation_block="qsogen_smc",
            agn_ebv=0.0,
        ),
    ),
    (
        "GRAHSP BBB + SKIRTOR torus + SMC",
        "tab:green",
        dict(
            agn_disc_block="grahsp_sbpl",
            agn_lines_block="grahsp",
            agn_feii_block="grahsp",
            agn_torus_block="skirtor",
            agn_attenuation_block="smc_prevot",
            agn_grahsp_ebv=0.0,
            agn_grahsp_ebv_agn=0.0,
            agn_tau_skirtor=7.0,
            agn_torus_frac=0.5,
            agn_attenuation_ebv=0.1,
        ),
    ),
    (
        "multicolor disc + Nenkova torus",
        "tab:red",
        dict(
            agn_disc_block="multicolor",
            agn_torus_block="nenkova",
            agn_log_mbh=8.0,
            agn_log_ledd=-1.0,
            agn_tau=30.0,
            agn_torus_frac=0.5,
        ),
    ),
]

fig, ax = plt.subplots(figsize=(8.5, 5.5))

for label, color, kw in RECIPES:
    l_nu = np.asarray(composable_agn_l_nu(wave_aa, **COMMON, **kw))
    nu_l_nu = l_nu * (2.99792458e18 / np.asarray(wave_aa))
    nu_l_nu = np.where(nu_l_nu > 0, nu_l_nu, np.nan)
    ax.loglog(wave_um, nu_l_nu, lw=2.0, color=color, label=label)

ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"$\nu L_\nu$ [erg s$^{-1}$]")
ax.set_xlim(5e-3, 1e2)
ax.set_ylim(1e43, 1e47)
ax.legend(loc="lower center", fontsize=9, frameon=False, ncol=1)
ax.set_title(
    r"Composable AGN recipes ($\log_{10}(L_\mathrm{bol}/L_\odot) = 12$): "
    "same call site, four selector tuples"
)
fig.tight_layout()